# 什么是 装饰器

> 1. 装饰器本质：一个函数，接收一个函数作为参数，返回一个新的函数
> 2. 使用方式——加上 `@decorator_name`

In [ ]:
def decorator_function(original_function):
    def wrapper(*args, **kwargs):
        # 调用前
        print("执行前")

        result = original_function(*args, **kwargs)

        # 调用后
        print("执行后")

        return result
    return wrapper

@decorator_function
def target_function():
    print("原函数执行")

target_function()

In [ ]:
def target_function():
    print("原函数执行")

target_function = decorator_function(target_function)
target_function()

- 目的——实现主要业务和辅助业务的分离
> 1. 比如主要业务是计算——可以做加减乘除以及更加复杂的运算
> 2. 辅助业务——打印函数执行所花费的时间
> 3. 以下面这个例子为例，通过装饰器就可以不用在运算函数里面去实现时间测量了
> 4. 所以说，装饰器的作用就是——增强原函数的功能（比如让计算函数具有实现时间测量的功能）

In [10]:
import time
def time_logger(calculate_func):
    def wrapper(*args, **kwargs):
        start = time.time()
        result = calculate_func(*args, **kwargs)
        time.sleep(1)
        end = time.time()
        print(f"花费了{end-start}s")
        return result
    return wrapper
@time_logger
def add(x, y):
    return x + y
@time_logger
def sub(x, y):
    return x - y
print(add(1,1))
print(sub(1,2))

花费了1.0050301551818848s
2
花费了1.0050299167633057s
-1


- 再来一些例子辅助理解为什么要用装饰器

In [11]:
def make_counter(start=0):
    # count 是外部函数的变量
    count = [start]

    def counter():
        # 用列表而不是普通变量，原因见下文解释
        current = count[0]
        count[0] += 1
        return current

    return counter

# 创建两个独立的计数器
counter_a = make_counter(0)
counter_b = make_counter(100)

# 各自独立计数
# 输出：A: 0, 1, 2
print(f"A: {counter_a()}, {counter_a()}, {counter_a()}")
# 输出：B: 100, 101, 102
print(f"B: {counter_b()}, {counter_b()}, {counter_b()}")

A: 0, 1, 2
B: 100, 101, 102


- `@functools.wraps(func)` 让装饰器保留`__name__`和`__doc__`字段

In [15]:
def decorator(func):
    def wrapper(*args, **kwargs):
        """
这是装饰器的注释文档
        """
        print("hello")
        res = func(*args, **kwargs)
        print("bye")
        return res
    return wrapper
@decorator
def add(x, y):
    """
    这是一个加法函数
    :param x:
    :param y:
    :return:
    """
    return x + y
print(add.__doc__)
print(add.__name__)


这是装饰器的注释文档
        
wrapper


In [16]:
import functools
def decorator(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        """
这是装饰器的注释文档
        """
        print("hello")
        res = func(*args, **kwargs)
        print("bye")
        return res
    return wrapper


@decorator
def add(x, y):
    """
这是加法函数的注释文档
    """
    return x + y


print(add.__doc__)
print(add.__name__)



这是加法函数的注释文档
    
add


### 带参数的装饰器

- 实现方式——在装饰器外层再套一个函数

In [17]:
import functools

def repeat(n):
    """让被装饰的函数重复执行 n 次"""
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            for i in range(n):
                result = func(*args, **kwargs)
            return result
        return wrapper
    return decorator

@repeat(3)
def say_hello(name):
    print(f"你好, {name}!")

# say_hello 会被执行 3 次
say_hello("Alice")

你好, Alice!
你好, Alice!
你好, Alice!


- 等价写法——先调用外层返回 `decoraotr` 然后在传递函数参数
- `say_hello = repeat(3)(say_hello)`

In [18]:
def say_hello(name):
    print(f"你好, {name}!")
say_hello = repeat(3)(say_hello)
# say_hello 会被执行 3 次
say_hello("Alice")

你好, Alice!
你好, Alice!
你好, Alice!
